The same merged dataset used in the EDA section is recreated here by loading the two CSV files, cleaning game names, and merging them based on title.

In [8]:
import pandas as pd
from scipy.stats import ttest_ind

# Load datasets
file1 = pd.read_csv("bestSelling_games.csv")
file2 = pd.read_csv("steam_games_2026.csv")

# Clean game names for matching
file1["game_name"] = file1["game_name"].str.lower().str.strip()
file2["Name"] = file2["Name"].str.lower().str.strip()

# Merge datasets
df = pd.merge(file1, file2, left_on="game_name", right_on="Name", how="inner")

First Hypothesis:

Null Hypothesis:
There is no significant difference in average estimated downloads between free and paid games.

Alternative Hypothesis:
There is a significant difference in average estimated downloads between free and paid games.

In [9]:
from scipy.stats import ttest_ind

free = df[df["price"] == 0]["estimated_downloads"]
paid = df[df["price"] > 0]["estimated_downloads"]

t_stat, p_value = ttest_ind(free, paid)

print("p-value:", p_value)

p-value: 7.736606989726841e-08


To examine whether free and paid games differ in popularity, an independent t-test was conducted.

The test produced a p-value of 7.73 × 10⁻⁸.

Since this value is significantly less than 0.05, the null hypothesis is rejected. This indicates that there is a statistically significant difference between free and paid games in terms of downloads.

This result supports the earlier analysis, where free games showed substantially higher download counts.

Second Hypothesis:

Null Hypothesis: Genre and downloads are independent

Alternative Hypothesis: Genre affects downloads

In [10]:
df["download_category"] = pd.cut(df["estimated_downloads"],
                                bins=[0, 1e6, 1e7, 1e8],
                                labels=["Low", "Medium", "High"])

In [11]:
from scipy.stats import chi2_contingency

table = pd.crosstab(df["Primary_Genre"], df["download_category"])
chi2, p, _, _ = chi2_contingency(table)

print("p-value:", p)

p-value: 0.1501983131316525


A chi-square test was conducted to examine the relationship between genre and download category. The p-value was 0.1502, which is greater than 0.05. Therefore, we fail to reject the null hypothesis.

This suggests that there is no statistically significant relationship between genre and download category.

However, exploratory data analysis indicates that some genres (such as Action) tend to have higher average downloads. This difference may not be statistically significant due to variability within genres and the categorization of downloads.

Third Hypothesis:

Null Hypothesis:
There is no significant relationship between review score and estimated downloads.

Alternative Hypothesis:
There is a significant relationship between review score and estimated downloads.

In [12]:
from scipy.stats import pearsonr
import numpy as np

corr_review, p_value_review = pearsonr(
    df["Review_Score_Pct"],
    np.log1p(df["estimated_downloads"])
)

print("Correlation:", corr_review)
print("p-value:", p_value_review)

Correlation: 0.04501275157394749
p-value: 0.3551730930225795


The Pearson correlation test produced a correlation coefficient of 0.0450 and a p-value of 0.3552.

Since the p-value is greater than 0.05, we fail to reject the null hypothesis. This means that there is no statistically significant relationship between review score and log-transformed estimated downloads.

The correlation value is also very close to 0, indicating that the relationship is extremely weak. Therefore, review score alone does not appear to be an important factor in explaining game popularity.

Fourth Hypothesis:

Null Hypothesis:
There is no significant relationship between game price and estimated downloads.

Alternative Hypothesis:
There is a significant relationship between game price and estimated downloads.

In [13]:
corr_price, p_value_price = pearsonr(
    df["price"],
    np.log1p(df["estimated_downloads"])
)

print("Correlation:", corr_price)
print("p-value:", p_value_price)

Correlation: -0.07165449303738937
p-value: 0.1407525113192543


The Pearson correlation test produced a correlation coefficient of -0.0717 and a p-value of 0.1408.

Since the p-value is greater than 0.05, we fail to reject the null hypothesis. This means that there is no statistically significant linear relationship between game price and log-transformed estimated downloads.

The negative correlation suggests a slight tendency for higher-priced games to have lower downloads, but the relationship is very weak and not statistically significant. Therefore, price as a continuous variable does not strongly explain download differences.

Overall, the hypothesis testing results show mixed findings.

The independent t-test showed that free and paid games differ significantly in terms of estimated downloads. This suggests that price accessibility has an important relationship with game popularity when games are grouped as free or paid.

However, the chi-square test did not show a statistically significant relationship between genre and download category. Although the EDA showed that some genres, such as Action, had higher average downloads, this relationship was not statistically strong in the chi-square test.

The Pearson correlation tests also showed that review score and price do not have statistically significant linear relationships with log-transformed estimated downloads. Both correlation values were very close to zero, meaning that these variables alone are weak indicators of popularity.

In conclusion, free games tend to perform better in terms of downloads, but Steam game popularity cannot be fully explained by price, genre, or review score alone. Popularity is likely influenced by more complex factors such as marketing, brand recognition, community activity, game visibility, and player trends.